In [ ]:
import os
from dotenv import load_dotenv
from openai import OpenAI
from pydantic import BaseModel
from langchain.output_parsers import PydanticOutputParser
from typing import List
import json

class SecurityAnalysis(BaseModel):
    """
    Security analysis model for RTL code, containing probability assessments from GPT-4

    Example:
    {
        "security_mechanisms": ["state_machine", "redundancy", "checksum", "encryption", "others"],
        "protection_mechanisms": ["fault_detection", "time_protection", "masking", "others"],
        "vulnerabilities": ["state_transition", "side_channel", "fault_injection", "time_side_channel", "others"],
        "exploit_methods": ["fault_attack", "energy_analysis", "detection", "replay", "others"],
        "security_score": 0.85
    }
    """
    security_mechanisms: List[str]
    protection_mechanisms: List[str]
    vulnerabilities: List[str]
    exploit_methods: List[str]
    security_score: float

def read_rtl_file(file_path: str) -> str:
    """Read RTL file content."""
    with open(file_path, 'r') as f:
        return f.read()

def create_rtl_analysis_prompt(protected_rtl: str, unprotected_rtl: str, parser: PydanticOutputParser) -> str:
    """Create analysis prompt for GPT-4."""
    base_prompt = """
    As an RTL code analysis expert, please analyze the following protected and unprotected RTL code differences and output the analysis results in the specified format:

    1. Security Mechanisms Analysis:
    - Identify security mechanisms used in protected code
    - Evaluate effectiveness of each mechanism
    - Provide list of security mechanisms [state_machine, redundancy, checksum, encryption, others]

    2. Protection Methods Analysis:
    - Identify specific protection implementation methods
    - Evaluate protection method strength
    - Provide list of protection methods [fault_detection, time_protection, masking, others]

    3. Vulnerability Analysis:
    - Identify potential vulnerabilities in unprotected code
    - Evaluate vulnerability severity
    - Provide list of vulnerability types [state_transition, side_channel, fault_injection, time_side_channel, others]

    4. Attack Methods Analysis:
    - Identify possible attack vectors
    - Evaluate attack feasibility
    - Provide list of attack methods [fault_attack, energy_analysis, detection, replay, others]

    5. Overall Security Score:
    - Provide a comprehensive security score between 0-1

    Protected Code:
    {protected_code}

    Unprotected Code:
    {unprotected_code}

    Please output the analysis results in the following JSON format:
    {format_instructions}

    Note:
    1. All mechanisms, methods, and vulnerabilities should be output as lists
    2. Each list element should be a string
    3. security_score should be a float between 0-1
    """

    return base_prompt.format(
        protected_code=protected_rtl,
        unprotected_code=unprotected_rtl,
        format_instructions=parser.get_format_instructions()
    )

def analyze_rtl_code(protected_rtl_path: str, unprotected_rtl_path: str, openai_api_key: str) -> dict:
    """
    Analyze RTL code using GPT-4 and return structured results.

    Args:
        protected_rtl_path (str): Path to protected RTL file
        unprotected_rtl_path (str): Path to unprotected RTL file
        openai_api_key (str): OpenAI API key

    Returns:
        dict: Structured analysis results
    """
    # Initialize OpenAI client
    client = OpenAI(api_key=openai_api_key)

    # Create output parser
    parser = PydanticOutputParser(pydantic_object=SecurityAnalysis)

    # Read RTL files
    protected_rtl = read_rtl_file(protected_rtl_path)
    unprotected_rtl = read_rtl_file(unprotected_rtl_path)

    # Create analysis prompt
    analysis_prompt = create_rtl_analysis_prompt(protected_rtl, unprotected_rtl, parser)

    # Use GPT-4 to generate analysis
    response = client.chat.completions.create(
        model="gpt-4",
        messages=[
            {"role": "system", "content": "You are an expert RTL code security analyst."},
            {"role": "user", "content": analysis_prompt}
        ],
        temperature=0.7,
        max_tokens=1000
    )

    # Parse and return results
    try:
        analysis_result = parser.parse(response.choices[0].message.content)
        return analysis_result.model_dump()
    except Exception as e:
        print(f"Error parsing GPT-4 response: {str(e)}")
        return None

def main():
    """Main function to demonstrate usage."""
    # Load environment variables
    load_dotenv()
    openai_api_key = os.getenv('OPENAI_API_KEY')

    if not openai_api_key:
        raise ValueError("OPENAI_API_KEY not found in environment variables")

    # Example paths - adjust as needed
    base_path = os.path.join('data', 'sample_data', 'BUILD', 'GCC_Plugin_Output', 'bit_count')
    protected_rtl_path = os.path.join(base_path, 'RTL_Protected.txt')
    unprotected_rtl_path = os.path.join(base_path, 'RTL.txt')

    # Analyze RTL code
    results = analyze_rtl_code(protected_rtl_path, unprotected_rtl_path, openai_api_key)

    if results:
        print("Analysis Results:")
        print(json.dumps(results, indent=2, ensure_ascii=False))
    else:
        print("Failed to analyze RTL code")

if __name__ == "__main__":
    main()

: 